# Scd Type 1

In [0]:
%sql
select * from dim_users;

user_id,user_name,city
101,Alice,New York


In [0]:
%sql
-- Load new data into staging
INSERT INTO stg_users VALUES 
(101, 'Alice', 'Los Angeles'), -- Existing user, new city
(102, 'Bob', 'Chicago');       -- New user


num_affected_rows,num_inserted_rows
2,2


In [0]:
%sql
merge into dim_users as trg using 
stg_users as src on trg.user_id=src.user_id
when matched then update set *
when not matched then insert *

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
2,1,0,1


In [0]:
%sql
select  * from dim_users;

user_id,user_name,city
101,Alice,Los Angeles
102,Bob,Chicago


# Scd type 2

In [0]:
%sql
CREATE TABLE scdtyp2_source
(
  prod_id INT,
  prod_name STRING,
  prod_cat STRING,
  processDate DATE
)

In [0]:
%sql
INSERT INTO scdtyp2_source
VALUES
(1,'prod1','cat1',CURRENT_DATE()),
(2,'prod2','cat2',CURRENT_DATE()),
(3,'prod3','cat3',CURRENT_DATE())

num_affected_rows,num_inserted_rows
3,3


In [0]:
%sql
CREATE TABLE scdtype2_table
(
  prod_id INT,
  prod_name STRING,
  prod_cat STRING,
  processDate DATE,
  start_date DATE,
  end_date DATE,
  is_current STRING
)

In [0]:
spark.sql(f"select *,current_date as start_date,'9999-01-01' as end_date,'Y' as is_current from scdtyp2_source").createOrReplaceTempView('source')

## Merge1: this merge will check if any data in target table updated in the source table ,if yes then it mark that old record as expired

In [0]:
%sql
merge into scdtype2_table as trg
 using source as src 
 on trg.prod_id=src.prod_id
  and trg.is_current='Y'
 when matched and(
 trg.prod_name <> src.prod_name or
  trg.prod_cat <> src.prod_cat 
 )then update set trg.end_date=src.start_date,
 trg.is_current='N'



num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
1,1,0,0


## 

## Merge2: this merge will insert row [new row/updated row] as new row  

In [0]:
%sql
merge into scdtype2_table as trg 
using source as src
on trg.prod_id=src.prod_id
and trg.is_current='Y'
when not matched then insert
(
  prod_id,
  prod_name,
  prod_cat,
  start_date,
  end_date,
  is_current
) VALUES (
  src.prod_id,
  src.prod_name,
  src.prod_cat,
  src.start_date,
  src.end_date,
  src.is_current
)

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
2,0,0,2


In [0]:
%sql
select * from scdtype2_table

prod_id,prod_name,prod_cat,processDate,start_date,end_date,is_current
1,prod1,cat1,2026-08-14,2026-08-14,9999-01-01,Y
2,prod2,cat2,2026-08-14,2026-08-14,9999-01-01,Y
3,prod3,cat3,2026-08-14,2026-08-14,9999-01-01,Y


In [0]:
%sql
truncate table scdtyp2_source

In [0]:
%sql
insert into scdtyp2_source values(1,'prod1','New_cat1',current_date());
insert into scdtyp2_source values(4,'prod1','cat4',current_date());

num_affected_rows,num_inserted_rows
1,1


In [0]:
%sql
select * from scdtype2_table

prod_id,prod_name,prod_cat,processDate,start_date,end_date,is_current
2,prod2,cat2,2026-08-14,2026-08-14,9999-01-01,Y
3,prod3,cat3,2026-08-14,2026-08-14,9999-01-01,Y
1,prod1,cat1,2026-08-14,2026-08-14,2026-08-14,N
1,prod1,New_cat1,2026-08-14,2026-08-14,9999-01-01,Y
4,prod1,cat4,2026-08-14,2026-08-14,9999-01-01,Y


# Scd type 3


In [0]:
%sql
truncate table scdtyp2_source

In [0]:
%sql
create table scdtype3_table (
  prod_id           int,
  prod_name         string,
  prod_cat          string,
  prod_cat_previous string,   -- tracks last value before change
  prod_cat_change_date date,  -- when the change happened
  processDate       date
);

In [0]:
%sql
insert into scdtyp2_source values(1,'prod1','cat1',current_date());
insert into scdtyp2_source values(4,'prod1','cat4',current_date());

num_affected_rows,num_inserted_rows
1,1


In [0]:
%sql
merge into scdtype3_table as trg 
using scdtyp2_source as src 
on trg.prod_id=src.prod_id
when matched and(trg.prod_cat <> src.prod_cat)
then update set
trg.prod_cat_previous=trg.prod_cat,
trg.prod_cat=src.prod_cat,
trg.prod_name=src.prod_name,
trg.prod_cat_change_date=current_date()



when not matched then insert 
(
  prod_id         ,
  prod_name         ,
  prod_cat          ,
  prod_cat_previous ,  -- tracks last value before change
  prod_cat_change_date ,  -- when the change happened
  processDate       

)values(
    src.prod_id,
    src.prod_name,
    src.prod_cat,
    null,
    null,
    src.processDate
)


num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
4,2,0,2


In [0]:
%sql
select * from scdtype3_table;

prod_id,prod_name,prod_cat,prod_cat_previous,prod_cat_change_date,processDate
1,prod1,cat1,null,null,2026-08-14
4,prod1,cat4,null,null,2026-08-14


In [0]:
%sql
insert into scdtyp2_source values(1,'prod1y','new_cat',current_date());
insert into scdtyp2_source values(4,'prod1x','new_cat',current_date());
insert into scdtyp2_source values(22,'prod1','cat1',current_date());
insert into scdtyp2_source values(44,'prod1','cat4',current_date());

num_affected_rows,num_inserted_rows
1,1


In [0]:
%sql
select * from scdtype3_table;

prod_id,prod_name,prod_cat,prod_cat_previous,prod_cat_change_date,processDate
4,prod1x,new_cat,cat4,2026-08-14,2026-08-14
1,prod1y,new_cat,cat1,2026-08-14,2026-08-14
44,prod1,cat4,null,null,2026-08-14
22,prod1,cat1,null,null,2026-08-14
